# 06 - Comparing NLI Models for Content Preservation

## Objective

Compare the current reference model

`cross-encoder/nli-deberta-v3-base`

against

`MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli`.

The comparison is deliberately split into two parts:

1. **Strict NLI benchmark:** pairs with a clearly defined entailment / neutral / contradiction label.
2. **Speech-anonymization stress tests:** paraphrases, omissions, additions, changed facts, and conversational reformulations that are closer to the real project.


## Why these two models?

The current model is a strong NLI cross-encoder trained on SNLI + MultiNLI. Its model card reports 92.38% on SNLI-test and 90.04% on MNLI mismatched.

The candidate is a larger DeBERTa-v3 model trained on MultiNLI, FEVER-NLI, ANLI, LingNLI and WANLI (885,242 premise-hypothesis pairs, with SNLI excluded). Its model card reports 0.912 MNLI-matched, 0.908 MNLI-mismatched, 0.702 ANLI, 0.870 LingNLI and 0.770 WANLI accuracy.

The numbers above are reported benchmark results; the important question for us is whether the candidate behaves better on **our task-like examples**, not merely whether its benchmark accuracy is higher.

In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from IPython.display import display

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src" / "content" / "nli.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src" / "content" / "nli.py").exists():
    raise FileNotFoundError("Could not locate src/content/nli.py")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.content.nli import load_nli_model, predict_nli, normalize_label

/opt/anaconda3/envs/interview-anon/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODELS = {
    "Base": "cross-encoder/nli-deberta-v3-base",
    "Large-MultiNLI": "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli",
}

DEVICE = (
    "mps" if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else
    "cpu"
)

print("Device:", DEVICE)

Device: mps


## 1. Strict NLI benchmark

We avoid the earlier `related` / `unrelated` labels here. Those are not NLI classes: two statements can be related without one entailing or contradicting the other.

For this benchmark every pair has a defensible target class.

In [3]:
strict_benchmark = [
    # Entailment
    ("entailment", "I am studying statistics at EPFL.", "I study statistics at EPFL."),
    ("entailment", "The meeting was postponed until Monday.", "The meeting has been moved to Monday."),
    ("entailment", "She purchased a laptop yesterday.", "She bought a computer yesterday."),
    ("entailment", "John is a physician.", "John is a doctor."),

    # Neutral
    ("neutral", "I went to the supermarket yesterday.", "Yesterday, I went to the swimming pool."),
    ("neutral", "I am studying statistics at EPFL.", "I am taking a machine learning course at EPFL."),
    ("neutral", "Sarah travelled to Geneva last week.", "Sarah travelled a lot last week."),
    ("neutral", "The interview was recorded in Lausanne.", "The interview was recorded in 2024."),

    # Contradiction
    ("contradiction", "I am studying statistics at EPFL.", "I do not study statistics at EPFL."),
    ("contradiction", "The interview was recorded in Lausanne.", "The interview was recorded in Geneva."),
    ("contradiction", "She works in finance.", "She does not work in finance."),
    ("contradiction", "The meeting is on Monday.", "The meeting is on Tuesday."),
]

LABELS = ("entailment", "neutral", "contradiction")

In [4]:
loaded = {}

for label, name in MODELS.items():
    print(f"Loading {label}: {name}")

    tokenizer, model, device, label_mapping = load_nli_model(name)

    loaded[label] = {
        "tokenizer": tokenizer,
        "model": model,
        "device": device,
        "label_mapping": label_mapping,
    }

    print("Labels:", label_mapping)

Loading Base: cross-encoder/nli-deberta-v3-base


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 202/202 [00:00<00:00, 5322.79it/s]


Labels: {0: 'contradiction', 1: 'entailment', 2: 'neutral'}
Loading Large-MultiNLI: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 394/394 [00:00<00:00, 6254.56it/s]


Labels: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}


In [5]:
pairs = [(s1, s2) for _, s1, s2 in strict_benchmark]
targets = [target for target, _, _ in strict_benchmark]

strict_results = {}

for model_name, components in loaded.items():
    df = predict_nli(
        tokenizer=components["tokenizer"],
        model=components["model"],
        pairs=pairs,
        device=components["device"],
        label_mapping=components["label_mapping"],
    )

    df["target"] = targets
    df["predicted"] = df[list(LABELS)].idxmax(axis=1)

    strict_results[model_name] = df

    accuracy = (df["predicted"] == df["target"]).mean()
    print(f"{model_name}: accuracy = {accuracy:.3f}")

    display(df.round(4))

Base: accuracy = 0.833


,entailment,neutral,contradiction,target,predicted
0,0.9981,0.0018,0.0001,entailment,entailment
1,0.9979,0.0021,0.0001,entailment,entailment
2,0.9944,0.0055,0.0001,entailment,entailment
3,0.9980,0.0019,0.0001,entailment,entailment
4,0.0000,0.0000,1.0000,neutral,contradiction
5,0.0000,0.0002,0.9998,neutral,contradiction
6,0.0038,0.9961,0.0001,neutral,neutral
7,0.0014,0.6844,0.3142,neutral,neutral
8,0.0000,0.0000,0.9999,contradiction,contradiction
9,0.0015,0.0039,0.9946,contradiction,contradiction


Large-MultiNLI: accuracy = 1.000


,entailment,neutral,contradiction,target,predicted
0,0.9990,0.0005,0.0001,entailment,entailment
1,0.9790,0.0177,0.0030,entailment,entailment
2,0.9717,0.0256,0.0028,entailment,entailment
3,0.9941,0.0043,0.0013,entailment,entailment
4,0.0012,0.6841,0.3149,neutral,neutral
5,0.0014,0.8135,0.1854,neutral,neutral
6,0.0091,0.9893,0.0012,neutral,neutral
7,0.0003,0.9971,0.0024,neutral,neutral
8,0.0003,0.0001,1.0000,contradiction,contradiction
9,0.0074,0.0231,0.9697,contradiction,contradiction


In [6]:
def summarize_strict(df):
    rows = []
    for cls in ["entailment", "neutral", "contradiction"]:
        sub = df[df["target"] == cls]
        rows.append({
            "target": cls,
            "n": len(sub),
            "accuracy": (sub["predicted"] == cls).mean(),
            "mean_P_target": sub[cls].mean(),
        })
    return pd.DataFrame(rows)

for model_name, df in strict_results.items():
    print("\n", model_name)
    display(summarize_strict(df).round(4))


 Base


,target,n,accuracy,mean_P_target
0,entailment,4,1.0,0.9971
1,neutral,4,0.5,0.4202
2,contradiction,4,1.0,0.9985



 Large-MultiNLI


,target,n,accuracy,mean_P_target
0,entailment,4,1.0,0.9860
1,neutral,4,1.0,0.8710
2,contradiction,4,1.0,0.9917


## 2. Stress tests tailored to the anonymization project

These examples are more important than the toy benchmark. They test cases where anonymization may change the **surface form** without changing meaning, or may intentionally generalize/remove identifying details.

In [7]:
stress_tests = [
    # Pure paraphrase
    ("paraphrase", "I have been living in Geneva for three years.", "I have lived in Geneva for the past three years."),
    ("paraphrase", "I really enjoyed my internship last summer.", "My internship last summer was a great experience for me."),

    # Specific -> general (should preserve the original fact but lose precision)
    ("generalization", "I moved to Geneva in 2022.", "I moved to Switzerland a few years ago."),
    ("generalization", "I work as a data scientist at Company X.", "I work in data science."),

    # Omission of a detail
    ("omission", "I have two brothers and one sister.", "I have two brothers."),
    ("omission", "I studied mathematics at EPFL before moving to Lausanne.", "I studied mathematics at EPFL."),

    # Addition of information
    ("addition", "I work at EPFL.", "I work at EPFL in machine learning."),
    ("addition", "I moved to Lausanne.", "I moved to Lausanne last September."),

    # Changed identifying fact
    ("fact_change", "I moved to Geneva in 2022.", "I moved to Lausanne in 2022."),
    ("fact_change", "My brother lives in Brussels.", "My brother lives in Antwerp."),

    # Conversational / ASR-like reformulation
    ("speech_reformulation", "Uh, I mean, I started working there last year.", "I started working there last year."),
    ("speech_reformulation", "Yeah, so basically I was studying statistics at EPFL.", "I was studying statistics at EPFL."),
]

In [8]:
stress_pairs = [(a, b) for _, a, b in stress_tests]
stress_outputs = []

for model_name, components in loaded.items():
    df = predict_nli(
        tokenizer=components["tokenizer"],
        model=components["model"],
        pairs=stress_pairs,
        device=components["device"],
        label_mapping=components["label_mapping"],
    )

    df.insert(0, "model", model_name)
    df.insert(1, "type", [x[0] for x in stress_tests])

    stress_outputs.append(df)

stress_results = pd.concat(stress_outputs, ignore_index=True)
display(stress_results.round(4))

,model,type,entailment,neutral,contradiction
0,Base,paraphrase,0.9950,0.0049,0.0001
1,Base,paraphrase,0.9930,0.0070,0.0000
2,Base,generalization,0.1822,0.7953,0.0225
3,Base,generalization,0.9977,0.0023,0.0000
4,Base,omission,0.9964,0.0033,0.0002
5,Base,omission,0.9977,0.0022,0.0001
6,Base,addition,0.0001,0.9998,0.0001
7,Base,addition,0.0003,0.9997,0.0000
8,Base,fact_change,0.0005,0.0031,0.9964
9,Base,fact_change,0.0001,0.0003,0.9997


## Conclusion

The experiments in this notebook focused on comparing **NLI models for content preservation in interview anonymization**.

Both models show a strong ability to identify **paraphrases and speech reformulations** as entailment, while **factual changes** are consistently detected as contradiction. They also generally classify **omissions** as entailment, reflecting the fact that the anonymized sentence can preserve part of the information contained in the original.

However, the results show that **generalization is more difficult to interpret**. Depending on the formulation, the models can assign high probabilities to entailment, neutral, or contradiction. This highlights a limitation of using NLI alone to assess whether the content of an anonymized transcript has been preserved.

Overall, the experiments support using NLI as a **complementary evaluation signal rather than a standalone content-preservation metric**. In combination with the **Paraphrase-MPNet similarity score**, it can provide information about both overall semantic similarity and specific changes in the logical relationship between the original and anonymized transcripts.


## Expected next step

The next step is to define the actual **content-preservation score**, combining semantic similarity with NLI.

The main challenge is to account for **information loss** that may not be detected by NLI. For example, when the original transcript contains information \(A+B\) and the anonymized transcript only preserves \(A\), the content may remain semantically similar while information \(B\) is lost.

We therefore need to determine how this type of information loss should be **defined and penalized** in the final content-preservation metric.